In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
# from google.colab import userdata

c:\Users\Vikash Kumar\Desktop\AG0847\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_folder_location = "tesla-annual-reports"

In [3]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [4]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=800,
    chunk_overlap=70
)

In [7]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [8]:
len(tesla_10k_chunks)

2365

# Database

In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [10]:

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Vikash Kumar\AppData\Local\Temp\ipykernel_13420\1925990503.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [11]:

# embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [12]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [13]:
chromadb_client

In [14]:
chromadb_client.heartbeat()

1780545176314393700

In [15]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [16]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

# Composing the response

In [17]:
from groq import Groq
client = Groq()

In [18]:
user_query = "What was the automotive revenue in 2021?"

In [19]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""



In [20]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


The automotive revenue in 2021 was **$47,232 million**.


# Improving RAG

In [21]:
import os
import chromadb

from langchain_chroma import Chroma
# from langchain_groq import ChatGroq



from langchain_core.documents import Document

# from google.colab import userdata

In [22]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [23]:
# !pip install langchain-community==0.3.19

from langchain_community.embeddings import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [24]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [25]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [26]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

# Hypothethical

In [27]:
hypothetical_questions_system_message = """
Generate a list of exactly 2 questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
The creation must stop when rate limit exceeded error is encountered so that it can be handled gracefully without giving any error.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [28]:
hypothetical_questions = []

In [29]:
len(tesla_10k_chunks)

2365

In [30]:
model_name = 'llama-3.1-8b-instant'

# Trying to create questions

In [31]:
# from tqdm import tqdm
# import time

# hypothetical_questions = []

# BATCH_SIZE = 5
# MAX_RETRIES = 5

# for batch_start in tqdm(
#     range(0, len(tesla_10k_chunks), BATCH_SIZE),
#     desc="Processing batches"
# ):
#     batch = tesla_10k_chunks[batch_start:batch_start + BATCH_SIZE]

#     batch_text = ""

#     for idx, document in enumerate(batch):
#         batch_text += f"\n\nDOCUMENT {idx + 1}\n"
#         batch_text += document.page_content

#     questions = ""

#     for attempt in range(MAX_RETRIES):
#         try:
#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": user_message_template.format(
#                             document=batch_text
#                         )
#                     }
#                 ],
#                 temperature=0
#             )

#             questions = response.choices[0].message.content.strip()
#             break

#         except Exception as e:
#             if "429" in str(e):
#                 wait_time = 2 ** attempt
#                 print(
#                     f"Rate limit reached. "
#                     f"Retrying in {wait_time} seconds..."
#                 )
#                 time.sleep(wait_time)
#             else:
#                 print(e)
#                 break

#     questions_list = questions.split("\n")

#     for question in questions_list:
#         question = question.strip()

#         if not question:
#             continue

#         for document in batch:
#             questions_metadata = {
#                 "parent_chunk_id": document.id,
#                 "parent_collection": "tesla_10k_collection"
#             }

#             hypothetical_questions.append(
#                 Document(
#                     page_content=question,
#                     metadata=questions_metadata
#                 )
#             )

In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm import tqdm
# import time

# BATCH_SIZE = 9       # Increase from 5
# MAX_WORKERS = 5       # Start with 5; increase if rate limits allow
# MAX_RETRIES = 5

# hypothetical_questions = []

# # Create batches
# batches = [
#     tesla_10k_chunks[i:i + BATCH_SIZE]
#     for i in range(0, len(tesla_10k_chunks), BATCH_SIZE)
# ]


# def process_batch(batch):
#     """
#     Sends one batch to the model and returns:
#     (batch, generated_questions)
#     """

#     batch_text = ""

#     for idx, document in enumerate(batch):
#         batch_text += f"\n\nDOCUMENT {idx + 1}\n"
#         batch_text += document.page_content

#     for attempt in range(MAX_RETRIES):
#         try:
#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": user_message_template.format(
#                             document=batch_text
#                         )
#                     }
#                 ],
#                 temperature=0
#             )

#             questions = response.choices[0].message.content.strip()

#             return batch, questions

#         except Exception as e:
#             if "429" in str(e):
#                 wait_time = 2 ** attempt
#                 print(
#                     f"Rate limit reached. "
#                     f"Retrying in {wait_time} seconds..."
#                 )
#                 time.sleep(wait_time)
#             else:
#                 print(f"Error: {e}")
#                 return batch, ""

#     return batch, ""


# # Process batches concurrently
# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

#     futures = [
#         executor.submit(process_batch, batch)
#         for batch in batches
#     ]

#     for future in tqdm(
#         as_completed(futures),
#         total=len(futures),
#         desc="Processing batches"
#     ):

#         batch, questions = future.result()

#         questions_list = [
#             q.strip()
#             for q in questions.split("\n")
#             if q.strip()
#         ]

#         for question in questions_list:

#             for document in batch:

#                 questions_metadata = {
#                     "parent_chunk_id": document.id,
#                     "parent_collection": "tesla_10k_collection"
#                 }

#                 hypothetical_questions.append(
#                     Document(
#                         page_content=question,
#                         metadata=questions_metadata
#                     )
#                 )

# print(
#     f"Generated {len(hypothetical_questions)} "
#     f"hypothetical questions."
# )

In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm import tqdm
# import time

# BATCH_SIZE = 9
# MAX_WORKERS = 5
# MAX_RETRIES = 5

# hypothetical_questions = []

# # Create batches
# batches = [
#     tesla_10k_chunks[i:i + BATCH_SIZE]
#     for i in range(0, len(tesla_10k_chunks), BATCH_SIZE)
# ]

# total_batches = len(batches)
# start_time = time.time()


# def process_batch(batch):
#     """
#     Sends one batch to the model and returns:
#     (batch, generated_questions)
#     """

#     batch_text = ""

#     for idx, document in enumerate(batch):
#         batch_text += f"\n\nDOCUMENT {idx + 1}\n"
#         batch_text += document.page_content

#     for attempt in range(MAX_RETRIES):
#         try:
#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": user_message_template.format(
#                             document=batch_text
#                         )
#                     }
#                 ],
#                 temperature=0
#             )

#             questions = response.choices[0].message.content.strip()

#             return batch, questions

#         except Exception as e:
#             if "429" in str(e):
#                 wait_time = 2 ** attempt

#                 print(
#                     f"\nRate limit reached. "
#                     f"Retrying in {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#             else:
#                 print(f"\nError: {e}")
#                 return batch, ""

#     return batch, ""


# # Process batches concurrently
# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

#     futures = [
#         executor.submit(process_batch, batch)
#         for batch in batches
#     ]

#     completed_batches = 0

#     for future in tqdm(
#         as_completed(futures),
#         total=len(futures),
#         desc="Processing batches"
#     ):

#         batch, questions = future.result()

#         completed_batches += 1

#         questions_list = [
#             q.strip()
#             for q in questions.split("\n")
#             if q.strip()
#         ]

#         for question in questions_list:

#             for document in batch:

#                 questions_metadata = {
#                     "parent_chunk_id": document.id,
#                     "parent_collection": "tesla_10k_collection"
#                 }

#                 hypothetical_questions.append(
#                     Document(
#                         page_content=question,
#                         metadata=questions_metadata
#                     )
#                 )

#         elapsed_time = time.time() - start_time

#         print(
#             f"\n✓ Batch {completed_batches}/{total_batches} completed"
#         )
#         print(
#             f"✓ Questions generated so far: "
#             f"{len(hypothetical_questions)}"
#         )
#         print(
#             f"✓ Elapsed time: "
#             f"{elapsed_time:.2f} seconds"
#         )

# print(
#     f"\nGenerated {len(hypothetical_questions)} "
#     f"hypothetical questions."
# )

# print(
#     f"Total execution time: "
#     f"{time.time() - start_time:.2f} seconds"
# )

Processing batches:   0%|          | 1/263 [00:00<00:34,  7.63it/s]


Error: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kt0xkpjhedcv9s3a5ffzsd8j` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6172, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

✓ Batch 1/263 completed
✓ Questions generated so far: 0
✓ Elapsed time: 0.24 seconds


Processing batches:   1%|          | 2/263 [00:00<02:13,  1.96it/s]


✓ Batch 2/263 completed
✓ Questions generated so far: 18
✓ Elapsed time: 1.02 seconds


Processing batches:   1%|          | 3/263 [00:43<1:26:17, 19.91s/it]


✓ Batch 3/263 completed
✓ Questions generated so far: 36
✓ Elapsed time: 44.03 seconds


Processing batches:   2%|▏         | 4/263 [01:23<1:59:57, 27.79s/it]


✓ Batch 4/263 completed
✓ Questions generated so far: 54
✓ Elapsed time: 83.89 seconds

Rate limit reached. Retrying in 1 seconds...

Rate limit reached. Retrying in 1 seconds...

Rate limit reached. Retrying in 1 seconds...

Rate limit reached. Retrying in 1 seconds...


In [32]:
from openai import OpenAI
import os

# os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

client = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1"
)

In [33]:
model_name = "meta/llama-3.1-70b-instruct"

# Generating Hypothetical Questions with NVIDIA

In [ ]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import os

# =========================
# NVIDIA NIM CONFIG
# =========================

client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

model_name = "meta/llama-3.1-70b-instruct"

# =========================
# PIPELINE CONFIG
# =========================

BATCH_SIZE = 9
MAX_WORKERS = 2
MAX_RETRIES = 5

hypothetical_questions = []

# =========================
# CREATE BATCHES
# =========================

batches = [
    tesla_10k_chunks[i:i + BATCH_SIZE]
    for i in range(0, len(tesla_10k_chunks), BATCH_SIZE)
]

total_batches = len(batches)

start_time = time.time()

# =========================
# PROCESS SINGLE BATCH
# =========================

def process_batch(batch):

    batch_text = ""

    for idx, document in enumerate(batch):

        batch_text += f"\n\nDOCUMENT {idx + 1}\n"
        batch_text += document.page_content

    for attempt in range(MAX_RETRIES):

        try:

            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": hypothetical_questions_system_message
                    },
                    {
                        "role": "user",
                        "content": user_message_template.format(
                            document=batch_text
                        )
                    }
                ],
                temperature=0,
                max_tokens=1000
            )

            questions = response.choices[0].message.content.strip()

            return batch, questions

        except Exception as e:

            print(f"\nError: {e}")

            if "429" in str(e):

                wait_time = min(
                    30,
                    5 * (attempt + 1)
                )

                print(
                    f"Rate limit reached. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                print(
                    f"Non-rate-limit error "
                    f"encountered."
                )

                return batch, ""

    return batch, ""

# =========================
# PROCESS BATCHES
# =========================

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = [
        executor.submit(
            process_batch,
            batch
        )
        for batch in batches
    ]

    completed_batches = 0

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing batches"
    ):

        batch, questions = future.result()

        completed_batches += 1

        questions_list = [
            q.strip()
            for q in questions.split("\n")
            if q.strip()
        ]

        for question in questions_list:

            for document in batch:

                questions_metadata = {
                    "parent_chunk_id": document.id,
                    "parent_collection": "tesla_10k_collection"
                }

                hypothetical_questions.append(
                    Document(
                        page_content=question,
                        metadata=questions_metadata
                    )
                )

        elapsed_time = (
            time.time() - start_time
        )

        avg_time_per_batch = (
            elapsed_time / completed_batches
        )

        remaining_batches = (
            total_batches -
            completed_batches
        )

        eta_seconds = (
            avg_time_per_batch *
            remaining_batches
        )

        print(
            f"\n✓ Batch "
            f"{completed_batches}/{total_batches}"
        )

        print(
            f"✓ Questions generated: "
            f"{len(hypothetical_questions)}"
        )

        print(
            f"✓ Elapsed: "
            f"{elapsed_time:.2f}s"
        )

        print(
            f"✓ ETA: "
            f"{eta_seconds/60:.1f} min"
        )

# =========================
# FINAL SUMMARY
# =========================

total_time = (
    time.time() - start_time
)

print(
    f"\nGenerated "
    f"{len(hypothetical_questions)} "
    f"hypothetical questions."
)

print(
    f"Total execution time: "
    f"{total_time:.2f} seconds"
)

Processing batches:   0%|          | 1/263 [00:51<3:46:44, 51.92s/it]


✓ Batch 1/263
✓ Questions generated: 18
✓ Elapsed: 51.94s
✓ ETA: 226.8 min


Processing batches:   1%|          | 2/263 [01:15<2:34:28, 35.51s/it]


✓ Batch 2/263
✓ Questions generated: 36
✓ Elapsed: 75.97s
✓ ETA: 165.2 min


Processing batches:   1%|          | 3/263 [01:21<1:35:20, 22.00s/it]

...


✓ Batch 261/263
✓ Questions generated: 4230
✓ Elapsed: 3790.32s
✓ ETA: 0.5 min


Processing batches: 100%|█████████▉| 262/263 [1:03:19<00:10, 10.98s/it]


✓ Batch 262/263
✓ Questions generated: 4248
✓ Elapsed: 3799.08s
✓ ETA: 0.2 min


Processing batches: 100%|██████████| 263/263 [1:03:20<00:00, 14.45s/it]


✓ Batch 263/263
✓ Questions generated: 4262
✓ Elapsed: 3800.50s
✓ ETA: 0.0 min

Generated 4262 hypothetical questions.
Total execution time: 3800.51 seconds


In [ ]:
import json

output_data = []

for doc in hypothetical_questions:
    output_data.append({
        "page_content": doc.page_content,
        "metadata": doc.metadata
    })

with open("hypothetical_questions.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output_data)} records.")

Saved 4262 records.


# Another Approach

In [44]:
for idx, doc in enumerate(tesla_10k_chunks):
    if "chunk_id" not in doc.metadata:
        doc.metadata["chunk_id"] = str(idx)

# Third Approach

In [32]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from langchain.schema import Document
import json
import time
import os

In [33]:


# =====================================================
# NVIDIA NIM CONFIG
# =====================================================

client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

model_name = "meta/llama-3.1-70b-instruct"

In [34]:

# =====================================================
# PROMPTS
# =====================================================

hypothetical_questions_system_message = """
You are generating hypothetical questions for a RAG system.

Generate EXACTLY 2 distinct questions that can be answered using the document.

Rules:
- Generate exactly 2 questions.
- One question per line.
- No numbering.
- No bullet points.
- No explanations.
- No introductory text.
- No concluding text.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [35]:
# =====================================================
# CONFIG
# =====================================================

MAX_WORKERS = 2
MAX_RETRIES = 5

hypothetical_questions = []

# =====================================================
# ENSURE CHUNK IDS EXIST
# =====================================================

for idx, doc in enumerate(tesla_10k_chunks):

    if "chunk_id" not in doc.metadata:
        doc.metadata["chunk_id"] = str(idx)

print(
    f"Assigned chunk_ids to "
    f"{len(tesla_10k_chunks)} chunks"
)

print(
    "\nSample chunk metadata:\n",
    tesla_10k_chunks[0].metadata
)


Assigned chunk_ids to 2365 chunks

Sample chunk metadata:
 {'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2022-05-02T10:10:26+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10ka_20211231-gen.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1', 'chunk_id': '0'}


In [ ]:
# # =====================================================
# # PROCESS SINGLE DOCUMENT
# # =====================================================

# def process_document(document):

#     chunk_id = document.metadata.get("chunk_id")

#     if chunk_id is None:

#         print(
#             f"\nWARNING: Missing chunk_id"
#             f"\nMetadata: {document.metadata}"
#         )

#         chunk_id = "unknown"

#     chunk_id = str(chunk_id)

#     document_text = document.page_content

#     for attempt in range(MAX_RETRIES):

#         try:

#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content":
#                             hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content":
#                             user_message_template.format(
#                                 document=document_text
#                             )
#                     }
#                 ],
#                 temperature=0,
#                 max_tokens=150
#             )

#             questions = (
#                 response
#                 .choices[0]
#                 .message
#                 .content
#                 .strip()
#             )

#             return chunk_id, questions

#         except Exception as e:

#             print(f"\nError: {e}")

#             if "429" in str(e):

#                 wait_time = min(
#                     30,
#                     5 * (attempt + 1)
#                 )

#                 print(
#                     f"Rate limit reached. "
#                     f"Retrying in "
#                     f"{wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#             else:

#                 return chunk_id, ""

#     return chunk_id, ""

# # =====================================================
# # GENERATE QUESTIONS
# # =====================================================

# start_time = time.time()

# with ThreadPoolExecutor(
#     max_workers=MAX_WORKERS
# ) as executor:

#     futures = [
#         executor.submit(
#             process_document,
#             document
#         )
#         for document in tesla_10k_chunks
#     ]

#     completed_docs = 0
#     total_docs = len(tesla_10k_chunks)

#     for future in tqdm(
#         as_completed(futures),
#         total=total_docs,
#         desc="Generating Questions"
#     ):

#         chunk_id, questions = future.result()

#         completed_docs += 1

#         questions_list = [
#             q.strip()
#             for q in questions.split("\n")
#             if q.strip()
#         ]

#         # STRICTLY KEEP ONLY 2 QUESTIONS
#         questions_list = questions_list[:2]

#         # =================================================
#         # DEBUG OUTPUT
#         # =================================================

#         print("\n" + "=" * 80)
#         print(f"Parent Chunk ID: {chunk_id}")
#         print(f"Questions Returned: {len(questions_list)}")

#         for idx, question in enumerate(
#             questions_list,
#             start=1
#         ):
#             print(f"Q{idx}: {question}")

#         print("=" * 80)

#         # =================================================
#         # CREATE DOCUMENTS
#         # =================================================

#         for question in questions_list:

#             hypothetical_questions.append(
#                 Document(
#                     page_content=question,
#                     metadata={
#                         "parent_chunk_id":
#                             chunk_id,
#                         "parent_collection":
#                             "full_document_chunks"
#                     }
#                 )
#             )

#         elapsed = (
#             time.time() - start_time
#         )

#         avg_time = (
#             elapsed / completed_docs
#         )

#         eta = avg_time * (
#             total_docs - completed_docs
#         )

#         if completed_docs % 25 == 0:

#             print(
#                 f"\n✓ Documents Processed: "
#                 f"{completed_docs}/{total_docs}"
#             )

#             print(
#                 f"✓ Questions Generated: "
#                 f"{len(hypothetical_questions)}"
#             )

#             print(
#                 f"✓ ETA: "
#                 f"{eta/60:.1f} minutes"
#             )

# # =====================================================
# # SUMMARY
# # =====================================================

# total_time = (
#     time.time() - start_time
# )

# print(
#     f"\nGenerated "
#     f"{len(hypothetical_questions)} "
#     f"questions."
# )

# print(
#     f"Execution time: "
#     f"{total_time:.2f} seconds"
# )

# # =====================================================
# # VERIFY OUTPUT
# # =====================================================

# print("\n")
# print("=" * 80)
# print("FIRST 10 GENERATED DOCUMENTS")
# print("=" * 80)

# for doc in hypothetical_questions[:10]:

#     print(
#         f"parent_chunk_id="
#         f"{doc.metadata['parent_chunk_id']}"
#     )

#     print(
#         f"question="
#         f"{doc.page_content}"
#     )

#     print("-" * 80)

# # =====================================================
# # SAVE TO JSON
# # =====================================================

# output_data = []

# for doc in hypothetical_questions:

#     output_data.append(
#         {
#             "page_content":
#                 doc.page_content,
#             "metadata":
#                 doc.metadata
#         }
#     )

# with open(
#     "hypothetical_questions.json",
#     "w",
#     encoding="utf-8"
# ) as f:

#     json.dump(
#         output_data,
#         f,
#         indent=2,
#         ensure_ascii=False
#     )

# print(
#     f"\nSaved "
#     f"{len(output_data)} "
#     f"records to "
#     f"hypothetical_questions.json"
# )

# 2 questions per chunks

# 4th Try

In [31]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain.schema import Document
from tqdm import tqdm
import json
import time
import os

# =====================================================
# NVIDIA NIM CONFIG
# =====================================================

client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

# model_name = "meta/llama-3.1-70b-instruct"
model_name = "meta/llama-3.1-8b-instruct"

# =====================================================
# CONFIG
# =====================================================

BATCH_SIZE = 50
MAX_WORKERS = 1
MAX_RETRIES = 5

hypothetical_questions = []

# =====================================================
# ENSURE CHUNK IDS EXIST
# =====================================================

for idx, doc in enumerate(tesla_10k_chunks):

    if "chunk_id" not in doc.metadata:
        doc.metadata["chunk_id"] = str(idx)

print(
    f"Total Chunks: {len(tesla_10k_chunks)}"
)

# =====================================================
# PROMPTS
# =====================================================

hypothetical_questions_system_message = """
You are generating hypothetical questions for a RAG system.

For EACH document, generate EXACTLY 2 questions.

Return in the following format:

DOCUMENT_ID: <id>
Question 1
Question 2

DOCUMENT_ID: <id>
Question 1
Question 2

Rules:
- Exactly 2 questions per document.
- One question per line.
- No numbering.
- No bullet points.
- No explanations.
- Preserve DOCUMENT_ID exactly.
"""

# =====================================================
# CREATE BATCHES
# =====================================================

batches = [
    tesla_10k_chunks[i:i + BATCH_SIZE]
    for i in range(
        0,
        len(tesla_10k_chunks),
        BATCH_SIZE
    )
]

print(
    f"Total Batches: {len(batches)}"
)

# =====================================================
# PROCESS BATCH
# =====================================================

def process_batch(batch):

    batch_text = ""

    for document in batch:

        chunk_id = str(
            document.metadata["chunk_id"]
        )

        batch_text += (
            f"\n\nDOCUMENT_ID: {chunk_id}\n"
        )

        batch_text += document.page_content

    for attempt in range(MAX_RETRIES):

        try:

            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content":
                            hypothetical_questions_system_message
                    },
                    {
                        "role": "user",
                        "content": batch_text
                    }
                ],
                temperature=0,
                max_tokens=1500
            )

            return (
                response
                .choices[0]
                .message
                .content
                .strip()
            )

        except Exception as e:

            print(f"\nError: {e}")

            if "429" in str(e):

                wait_time = min(
                    60,
                    10 * (attempt + 1)
                )

                print(
                    f"Rate limit reached. "
                    f"Retrying in "
                    f"{wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                return ""

    return ""

# =====================================================
# PARSE RESPONSE
# =====================================================

def parse_response(response_text):

    current_chunk_id = None

    for line in response_text.splitlines():

        line = line.strip()

        if not line:
            continue

        if line.startswith("DOCUMENT_ID:"):

            current_chunk_id = (
                line.replace(
                    "DOCUMENT_ID:",
                    ""
                )
                .strip()
            )

            continue

        if current_chunk_id:

            hypothetical_questions.append(
                Document(
                    page_content=line,
                    metadata={
                        "parent_chunk_id":
                            current_chunk_id,
                        "parent_collection":
                            "full_document_chunks"
                    }
                )
            )

# =====================================================
# GENERATE QUESTIONS
# =====================================================

start_time = time.time()

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = [
        executor.submit(
            process_batch,
            batch
        )
        for batch in batches
    ]

    completed = 0
    total_batches = len(batches)

    for future in tqdm(
        as_completed(futures),
        total=total_batches,
        desc="Generating Questions"
    ):

        response_text = future.result()

        parse_response(response_text)

        completed += 1

        elapsed = (
            time.time() - start_time
        )

        avg_time = (
            elapsed / completed
        )

        eta = avg_time * (
            total_batches - completed
        )

        print(
            f"\n✓ Batch "
            f"{completed}/{total_batches}"
        )

        print(
            f"✓ Questions Generated: "
            f"{len(hypothetical_questions)}"
        )

        print(
            f"✓ ETA: "
            f"{eta/60:.1f} minutes"
        )

# =====================================================
# SUMMARY
# =====================================================

total_time = (
    time.time() - start_time
)

print("\n" + "=" * 80)

print(
    f"Generated "
    f"{len(hypothetical_questions)} "
    f"questions"
)

print(
    f"Execution Time: "
    f"{total_time:.2f} seconds"
)

print("=" * 80)

# =====================================================
# VERIFY OUTPUT
# =====================================================

print("\nSample Output:\n")

for doc in hypothetical_questions[:10]:

    print(
        f"parent_chunk_id="
        f"{doc.metadata['parent_chunk_id']}"
    )

    print(
        f"question="
        f"{doc.page_content}"
    )

    print("-" * 80)

# =====================================================
# SAVE JSON
# =====================================================

output_data = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in hypothetical_questions
]

with open(
    "hypothetical_questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nSaved "
    f"{len(output_data)} records "
    f"to hypothetical_questions.json"
)

Total Chunks: 2365
Total Batches: 48


Generating Questions:   2%|▏         | 1/48 [00:03<02:41,  3.43s/it]


✓ Batch 1/48
✓ Questions Generated: 2
✓ ETA: 2.7 minutes


Generating Questions:   4%|▍         | 2/48 [00:21<09:03, 11.83s/it]


✓ Batch 2/48
✓ Questions Generated: 73
✓ ETA: 8.1 minutes


...


✓ Batch 46/48
✓ Questions Generated: 2252
✓ ETA: 0.6 minutes


Generating Questions:  98%|█████████▊| 47/48 [13:44<00:16, 16.21s/it]


✓ Batch 47/48
✓ Questions Generated: 2311
✓ ETA: 0.3 minutes


Generating Questions: 100%|██████████| 48/48 [13:48<00:00, 17.26s/it]


✓ Batch 48/48
✓ Questions Generated: 2341
✓ ETA: 0.0 minutes

Generated 2341 questions
Execution Time: 828.69 seconds

Sample Output:

parent_chunk_id=0
question=What is the primary purpose of the 2018 CEO Performance Award granted to Elon Musk?
--------------------------------------------------------------------------------
parent_chunk_id=0
question=What are the key milestones that Elon Musk must achieve in order to vest the 2018 CEO Performance Award?
--------------------------------------------------------------------------------
parent_chunk_id=50
question=What are the key components of the Director Compensation Policy at Tesla?
--------------------------------------------------------------------------------
parent_chunk_id=50
question=What are the implications of the Compensation Committee's decision to waive automatic grants of annual stock option awards to existing directors?
--------------------------------------------------------------------------------
parent_chunk_id=51
qu

# Old

In [ ]:


# # =====================================================
# # PROCESS SINGLE DOCUMENT
# # =====================================================

# def process_document(document):

#     chunk_id = str(
#         document.metadata["chunk_id"]
#     )

#     document_text = document.page_content

#     for attempt in range(MAX_RETRIES):

#         try:

#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": user_message_template.format(
#                             document=document_text
#                         )
#                     }
#                 ],
#                 temperature=0,
#                 max_tokens=150
#             )

#             questions = (
#                 response
#                 .choices[0]
#                 .message
#                 .content
#                 .strip()
#             )

#             return chunk_id, questions

#         except Exception as e:

#             print(f"\nError: {e}")

#             if "429" in str(e):

#                 wait_time = min(
#                     30,
#                     5 * (attempt + 1)
#                 )

#                 print(
#                     f"Rate limit reached. "
#                     f"Retrying in {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#             else:

#                 return chunk_id, ""

#     return chunk_id, ""

# # =====================================================
# # GENERATE QUESTIONS
# # =====================================================

# start_time = time.time()

# with ThreadPoolExecutor(
#     max_workers=MAX_WORKERS
# ) as executor:

#     futures = [
#         executor.submit(
#             process_document,
#             document
#         )
#         for document in tesla_10k_chunks
#     ]

#     completed_docs = 0
#     total_docs = len(tesla_10k_chunks)

#     for future in tqdm(
#         as_completed(futures),
#         total=total_docs,
#         desc="Generating Questions"
#     ):

#         chunk_id, questions = future.result()

#         completed_docs += 1

#         questions_list = [
#             q.strip()
#             for q in questions.split("\n")
#             if q.strip()
#         ]

#         # STRICT LIMIT
#         questions_list = questions_list[:2]

#         for question in questions_list:

#             hypothetical_questions.append(
#                 Document(
#                     page_content=question,
#                     metadata={
#                         "parent_chunk_id": chunk_id,
#                         "parent_collection":
#                             "full_document_chunks"
#                     }
#                 )
#             )

#         elapsed = (
#             time.time() - start_time
#         )

#         avg_time = elapsed / completed_docs

#         eta = avg_time * (
#             total_docs - completed_docs
#         )

#         if completed_docs % 25 == 0:

#             print(
#                 f"\n✓ Documents: "
#                 f"{completed_docs}/{total_docs}"
#             )

#             print(
#                 f"✓ Questions: "
#                 f"{len(hypothetical_questions)}"
#             )

#             print(
#                 f"✓ ETA: "
#                 f"{eta/60:.1f} min"
#             )

# # =====================================================
# # SUMMARY
# # =====================================================

# total_time = time.time() - start_time

# print(
#     f"\nGenerated "
#     f"{len(hypothetical_questions)} "
#     f"questions."
# )

# print(
#     f"Execution time: "
#     f"{total_time:.2f} seconds"
# )

# # =====================================================
# # SAVE TO JSON
# # =====================================================

# output_data = []

# for doc in hypothetical_questions:

#     output_data.append(
#         {
#             "page_content":
#                 doc.page_content,
#             "metadata":
#                 doc.metadata
#         }
#     )

# with open(
#     "hypothetical_questions.json",
#     "w",
#     encoding="utf-8"
# ) as f:

#     json.dump(
#         output_data,
#         f,
#         indent=2,
#         ensure_ascii=False
#     )

# print(
#     f"Saved "
#     f"{len(output_data)} "
#     f"records to hypothetical_questions.json"
# )

# Before another run

In [46]:
import json

with open("hypothetical_questions.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total records: {len(data)}")
print(data[:3])

Total records: 2341
[{'page_content': 'What is the primary purpose of the 2018 CEO Performance Award granted to Elon Musk?', 'metadata': {'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}}, {'page_content': 'What are the key milestones that Elon Musk must achieve in order to vest the 2018 CEO Performance Award?', 'metadata': {'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}}, {'page_content': 'What are the key components of the Director Compensation Policy at Tesla?', 'metadata': {'parent_chunk_id': '50', 'parent_collection': 'full_document_chunks'}}]


In [47]:
hypothetical_questions[0]

Document(metadata={'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}, page_content='What is the primary purpose of the 2018 CEO Performance Award granted to Elon Musk?')

In [48]:
print(retrieved_docs[0].metadata)
print(type(retrieved_docs[0].metadata["parent_chunk_id"]))

{'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}
<class 'str'>


In [49]:
all_chunk_ids = set()

for d in retrieved_docs:
    all_chunk_ids.add(
        str(d.metadata["parent_chunk_id"])
    )

retrieved_documents = vectorstore_persisted.get(
    ids=list(all_chunk_ids)
)

print(retrieved_documents["documents"])

[]


In [51]:
final_query_system = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.
 
User queries will be delimited by: <Question> and </Question>.
 
Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.
 
If the answer is not found in the context, respond "I don't know".
"""
 
final_query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>
 
<Question>
{question}
</Question>
"""
 
prompt = [
    {
        "role": "system",
        "content": final_query_system
    },
    {
        "role": "user",
        "content": final_query_user_message_template.format(
            context="\n".join(retrieved_documents['documents']),
            question=user_query
        )
    }
]
 
model_name="meta/llama-3.1-8b-instruct"
 
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )
 
    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = e
 
print(prediction)

I don't know.


In [38]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [39]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'hypothetical_questions', 'full_document_chunks']

In [40]:
hypothetical_questions_vectorstore.add_documents( 
    documents=hypothetical_questions
)

['d8629c91-f9f3-4973-96bd-d69fda4ba3b6',
 '28cc4563-2bdd-4e72-b1ea-85611526983c',
 'cba1ee48-543f-4049-b6c5-ce6709719bb9',
 '894e1706-4c8f-4a6d-86f8-1fba859ce544',
 '267b0f59-859a-4be8-b2fa-552ba31b4d7b',
 '290d1195-3c61-411c-b073-ccf49982361b',
 '0b642425-b2e0-4244-85c9-773af79dba2b',
 'f6dc5ffe-9446-418a-8b19-638fb112ea2c',
 'bde1720d-e397-494c-9ed1-00d36557acdb',
 'ff1401e1-0068-4a33-998d-6c800d464756',
 '9054570e-8c95-4059-a414-316546677ab0',
 '8fc87f1d-2284-4809-ac4b-9fa3491fab10',
 '1288cc89-5105-4d1b-8d71-bb31b604acb8',
 '1f974855-e1fa-411f-83ff-38aa334888db',
 '3cdc7a93-d31a-4303-b933-f9a7f4d6e427',
 'fc78b077-2c5b-44c2-b767-349c98229c6f',
 'c3385bf0-d4fa-4712-94c7-93f1a8a029b2',
 '0268fb6b-01f8-4ec2-8ff7-acc9b9187ec3',
 '6606de2e-93d3-42eb-b605-8c4763e603e9',
 '3d0c2697-fe5b-4f53-a820-ad7e90b8ba76',
 '67f6ad36-f949-488c-b541-bff201c5b3a6',
 '4338c507-799e-4164-a882-d244d8c66dec',
 'd4eaa761-6a44-4165-bbc2-52447685e196',
 '68580115-cc7e-4070-b979-2e64d187ed7d',
 'a80921a5-575e-

In [41]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [42]:
user_query = "Any specific fines levied on the company in 2022?"

In [43]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [44]:
hypothetical_questions_retrieved

[Document(id='784ea7c9-f761-453f-9221-9a41ef7c2634', metadata={'parent_chunk_id': '2301', 'parent_collection': 'full_document_chunks'}, page_content='What are the potential risks and consequences for Tesla if the government decides to pursue an enforcement action?'),
 Document(id='5c092703-4761-413e-876d-cd4951f49b88', metadata={'parent_chunk_id': '681', 'parent_collection': 'full_document_chunks'}, page_content='What could happen if Tesla is unable to manage the risks associated with its environmental regulations and aggregate civil penalties?'),
 Document(id='592388b3-058d-4786-a096-fa4f09b679ad', metadata={'parent_chunk_id': '681', 'parent_collection': 'full_document_chunks'}, page_content="What are the potential risks associated with Tesla's environmental regulations and aggregate civil penalties?"),
 Document(id='a9825507-3f07-40f9-8b1f-8b9c64f2777c', metadata={'parent_chunk_id': '1787', 'parent_collection': 'full_document_chunks'}, page_content='What is the potential material adv

In [45]:
retrived_documents = vectorstore.get(
    ids=list(set([d.metadata['parent_chunk_id'] for d in hypothetical_questions_retrieved]))
)

NameError: name 'vectorstore' is not defined

In [ ]:
print(retrived_documents)

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'data': None, 'metadatas': [], 'included': [<IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


In [ ]:
hypothetical_questions[0], hypothetical_questions[1], hypothetical_questions[2]

(Document(metadata={'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}, page_content='What is the primary purpose of the 2018 CEO Performance Award granted to Elon Musk?'),
 Document(metadata={'parent_chunk_id': '0', 'parent_collection': 'full_document_chunks'}, page_content='What are the key milestones that Elon Musk must achieve in order for the 2018 CEO Performance Award to vest?'),
 Document(metadata={'parent_chunk_id': '1', 'parent_collection': 'full_document_chunks'}, page_content="What is the current executive compensation program for Tesla's named executive officers, and how does it align with the company's long-term interests?"))

In [ ]:
question_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know, Vikash".
"""

question_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""



In [ ]:
question_system_message = """
You are an assistant for analyzing annual reports.

Instructions:
- Answer ONLY using the information provided in the context.
- Do not use outside knowledge.
- Do not mention the context, documents, or sources in your answer.
- If the answer cannot be determined from the provided information, respond exactly:
I don't know, Vikash
- Be concise but complete.
"""

In [ ]:
user_input = "Any specific fines levied on the company in 2022?"

In [ ]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [ ]:
final_context_documents1 = set(retrived_documents)

In [ ]:
prompts_new = [
    {"role": "system", "content": question_system_message},
    {"role": "user", "content": question_user_message_template.format(
        context=final_context_documents1, 
        question=user_input)}
]

In [ ]:
prompt = [
    {"role": "system", "content": query_expansion_system_message},
    {"role": "user", "content": user_message_template.format(question=user_input)}
]

In [ ]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nAny specific fines levied on the company in 2022?\n</Question>\n'}]

In [ ]:
query_expansions = client.chat.completions.create(model=model_name, 
                                                  messages=prompt, 
                                                  temperature=0)

In [ ]:
type(query_expansions.choices[0].message.content)

str

In [ ]:
print(query_expansions)

ChatCompletion(id='chatcmpl-3622d831-c230-465d-8675-4d9c06ea63e5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="What are the specific fines imposed on the company in 2022 as disclosed in the 10-K report?\nWhat fines were levied against the company in 2022 according to the company's annual report?\nAre there any notable fines or penalties incurred by the company in 2022 as stated in the 10-K filing?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1780501929, model='meta/llama-3.1-8b-instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=67, prompt_tokens=188, total_tokens=255, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=16)), nvext={'worker_id': {'prefill_worker_id': 7587895250560495977, 'prefill_dp_rank': 0, 'decode_w

In [ ]:
print(query_expansions.choices[-1].message.content)

What are the specific fines imposed on the company in 2022 as disclosed in the 10-K report?
What fines were levied against the company in 2022 according to the company's annual report?
Are there any notable fines or penalties incurred by the company in 2022 as stated in the 10-K filing?


In [ ]:
prompts_new

[{'role': 'system',
  'content': "\nYou are an assistant for analyzing annual reports.\n\nInstructions:\n- Answer ONLY using the information provided in the context.\n- Do not use outside knowledge.\n- Do not mention the context, documents, or sources in your answer.\n- If the answer cannot be determined from the provided information, respond exactly:\nI don't know, Vikash\n- Be concise but complete.\n"},
 {'role': 'user',
  'content': "\n<Context>\nHere are some documents that are relevant to the question mentioned below.\n{'metadatas', 'ids', 'embeddings', 'uris', 'data', 'included', 'documents'}\n</Context>\n\n<Question>\nAny specific fines levied on the company in 2022?\n</Question>\n"}]

In [ ]:
response = client.chat.completions.create(model=model_name, 
                                          messages=prompts_new, 
                                          temperature=0)

In [ ]:
print(response.choices[0].message.content)

I don't know, Vikash
